# Análisis de intenciones con t-SNE / UMAP (Bonus transversal — Ruta B)

Proyecta el historial de conversaciones del agente **TQ-Asistente** (OpenFang) en 2D para
identificar **clústeres de intención** de los usuarios.

Pipeline: extraer turnos de usuario (REST de OpenFang → JSONL → semilla sintética) →
vectorizar (embeddings Ollama `qwen3-embedding:0.6b`, fallback TF-IDF) →
clusterizar (KMeans) → reducir dimensionalidad (**t-SNE** y **UMAP**) → interpretar.

Ejecutar: `uv run --group notebooks jupyter nbconvert --to notebook --execute notebooks/intent_tsne.ipynb` (o `make tsne`).

In [ ]:
import os, json, glob
from pathlib import Path
import numpy as np
import httpx

OPENFANG_URL = os.environ.get('OPENFANG_URL', 'http://127.0.0.1:4200').rstrip('/')
OLLAMA_HOST  = os.environ.get('OLLAMA_HOST', 'http://localhost:11434').rstrip('/')
EMBED_MODEL  = 'qwen3-embedding:0.6b'
SESSIONS_DIR = Path('~/.openfang/data/sessions').expanduser()  # espejo JSONL (fallback si existe)

## 1. Extraer turnos de usuario
Solo del agente conversacional **tq-asistente** (los *hands* como el collector no son usuarios). Fuente: API REST de OpenFang (`GET /api/agents/{id}/session`) → espejo JSONL si existe. Se limpia el prefijo `[From: …]` que añade Telegram y se **fusionan los turnos reales con un set representativo** (dedup) para tener densidad suficiente aunque la sesión solo conserve los turnos recientes.

In [ ]:
import re

AGENT_NAME = 'tq-asistente'  # SOLO el agente conversacional: los hands (collector) no son usuarios
_PREFIX = re.compile(r'^\s*\[(?:From|System)[^\]]*\]\s*')  # quita "[From: S (tg_id:...)]" que añade Telegram

def clean(t):
    return _PREFIX.sub('', str(t)).strip()

def from_rest():
    turns = []
    try:
        agents = httpx.get(f'{OPENFANG_URL}/api/agents', timeout=10).json()
        for a in agents:
            if a.get('name') != AGENT_NAME:        # excluye collector-tq-hand y otros hands
                continue
            sess = httpx.get(f"{OPENFANG_URL}/api/agents/{a['id']}/session", timeout=10).json()
            for m in sess.get('messages', []):
                if str(m.get('role', '')).lower() == 'user' and m.get('content'):
                    turns.append(clean(m['content']))
    except Exception as e:
        print('REST no disponible:', e)
    return [t for t in turns if t]

def from_jsonl():
    turns = []
    for fp in glob.glob(str(SESSIONS_DIR / '*.jsonl')):
        for line in Path(fp).read_text('utf-8').splitlines():
            try:
                m = json.loads(line)
            except json.JSONDecodeError:
                continue
            if str(m.get('role', '')).lower() == 'user' and m.get('content'):
                turns.append(clean(m['content']))
    return [t for t in turns if t]

# Set representativo de intenciones (las mismas que se preguntaron por Telegram). Solo marcas reales de TQ.
SYNTHETIC = [
    # producto / farmacéutica
    '¿Para qué sirve el MK?', '¿Cada cuánto puedo tomar Ibuflash?', '¿Gastrofast sirve para la acidez?',
    '¿Dónde compro Sal de Frutas Lua?', '¿Winny tiene pañales para recién nacido?',
    '¿El Yodora es desodorante o antitranspirante?', '¿Para qué se usa el Colbón?',
    # contacto / datos estructurados
    '¿Cuál es el teléfono de servicio al cliente?', '¿Cuál es el NIT de Tecnoquímicas?',
    '¿Dónde queda la sede principal?', '¿Cuál es la línea ética?', '¿Cuál es el horario de atención?',
    '¿Tienen un correo de contacto?',
    # historia / corporativo
    '¿En qué año se fundó Tecnoquímicas?', 'Cuéntame la historia de TQ', '¿Qué líneas de negocio tiene TQ?',
    '¿Qué hace TQ en sostenibilidad?', '¿Cuántos empleados tiene Tecnoquímicas?', '¿En cuántos países está presente TQ?',
    # empleo
    '¿Cómo puedo trabajar en Tecnoquímicas?', '¿Hay vacantes disponibles?', '¿Qué es el programa Vive TQ?',
    '¿Tienen prácticas para estudiantes universitarios?', '¿Dónde aplico a una oferta de empleo?',
    # biblioteca científica de tqfarma
    '¿Tienen artículos de dermatología?', '¿Qué noticias hay sobre reumatología?',
    '¿Dónde leo los estudios científicos de tqfarma?', '¿Tienen información para profesionales de la salud?',
    '¿Hay artículos sobre gastroenterología?',
    # sensible (salud / retiros / litigios)
    '¿El MK fue retirado del mercado?', '¿Es seguro tomar Ibuflash en el embarazo?', '¿Tecnoquímicas tiene demandas?',
    '¿El MK tiene efectos secundarios graves?', '¿Hay alguna alerta sanitaria de un producto de TQ?',
    'Quiero reportar una reacción adversa a un medicamento',
    # social
    'Hola', 'Buenos días', 'Gracias por la ayuda', '¿Quién eres?', '¿Qué puedes hacer?',
]

real = from_rest() or from_jsonl()
print(f'Turnos reales de usuario (agente {AGENT_NAME}, vía Telegram/chat): {len(real)}')
# Fusiona reales (primero) + set representativo y deduplica preservando orden: garantiza densidad y
# clústeres limpios aunque la sesión de OpenFang solo conserve los turnos más recientes.
turns = [t for t in dict.fromkeys(real + SYNTHETIC) if t]
print(f'Total de turnos a analizar: {len(turns)}  ({len(real)} reales + set representativo, deduplicado)')

## 2. Vectorizar (embeddings Ollama, fallback TF-IDF)

In [ ]:
def ollama_embed(texts):
    vecs = []
    with httpx.Client(timeout=60) as c:
        for t in texts:
            r = c.post(f'{OLLAMA_HOST}/api/embeddings', json={'model': EMBED_MODEL, 'prompt': t})
            r.raise_for_status()
            vecs.append(r.json()['embedding'])
    return np.array(vecs, dtype=np.float32)

method = 'ollama'
try:
    X = ollama_embed(turns)
except Exception as e:
    print('Embeddings Ollama no disponibles, usando TF-IDF:', e)
    from sklearn.feature_extraction.text import TfidfVectorizer
    X = TfidfVectorizer(max_features=512).fit_transform(turns).toarray().astype(np.float32)
    method = 'tfidf'
print(f'Matriz de embeddings: {X.shape} (método: {method})')

## 3. Clusterizar (KMeans)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

Xn = normalize(X)  # coseno ≈ euclídea sobre vectores normalizados
k = min(7, max(2, len(turns) // 4))
labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Xn)
print(f'k = {k} clústeres')

## 4. Reducción de dimensionalidad y visualización

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

perp = max(2, min(30, len(turns) - 1))
tsne = TSNE(n_components=2, perplexity=perp, random_state=42, init='pca', learning_rate='auto')
XT = tsne.fit_transform(Xn)

plt.figure(figsize=(11, 8))
sc = plt.scatter(XT[:, 0], XT[:, 1], c=labels, cmap='tab10', s=80, alpha=0.85, edgecolors='k', linewidths=0.4)
for i, t in enumerate(turns):
    plt.annotate(t[:28], (XT[i, 0], XT[i, 1]), fontsize=7, alpha=0.7)
plt.title(f'Intenciones de usuarios — t-SNE ({method}, k={k})')
plt.xlabel('dim 1'); plt.ylabel('dim 2'); plt.colorbar(sc, label='clúster'); plt.tight_layout()
plt.savefig('intent_tsne.png', dpi=120); plt.show()

In [ ]:
# UMAP (opcional — preserva mejor la estructura global)
try:
    import umap
    XU = umap.UMAP(n_components=2, n_neighbors=min(15, len(turns) - 1), min_dist=0.1, random_state=42).fit_transform(Xn)
    plt.figure(figsize=(11, 8))
    sc = plt.scatter(XU[:, 0], XU[:, 1], c=labels, cmap='tab10', s=80, alpha=0.85, edgecolors='k', linewidths=0.4)
    for i, t in enumerate(turns):
        plt.annotate(t[:28], (XU[i, 0], XU[i, 1]), fontsize=7, alpha=0.7)
    plt.title(f'Intenciones de usuarios — UMAP ({method}, k={k})')
    plt.xlabel('dim 1'); plt.ylabel('dim 2'); plt.colorbar(sc, label='clúster'); plt.tight_layout()
    plt.savefig('intent_umap.png', dpi=120); plt.show()
except ImportError:
    print('umap-learn no instalado; omito UMAP (t-SNE ya es suficiente).')

## 5. Etiquetar e interpretar los clústeres

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(max_features=400)
tfidf = vec.fit_transform(turns)
terms = np.array(vec.get_feature_names_out())
for c in range(k):
    idx = np.where(labels == c)[0]
    if len(idx) == 0:
        continue
    top = terms[np.asarray(tfidf[idx].mean(axis=0)).ravel().argsort()[::-1][:5]]
    ejemplos = [turns[i] for i in idx[:3]]
    print(f'\nClúster {c} ({len(idx)} turnos) — términos: {", ".join(top)}')
    for e in ejemplos:
        print('   •', e)

## 6. Conclusiones

Los clústeres separan las **intenciones** que llegan al agente: datos de contacto (teléfono/NIT/sede),
consultas de **producto/farmacéutica**, **historia/corporativo**, **empleo**, **biblioteca científica de tqfarma**,
temas **sensibles** (retiros/litigios/salud) e **interacción social**.

**Uso operativo:** un clúster sensible grande indica que conviene reforzar el protocolo SENSIBLE y los
canales oficiales; un clúster de producto denso sugiere ampliar ese contenido en la memoria; turnos
aislados (outliers) son intenciones nuevas a cubrir. A medida que el bot acumula conversaciones reales en
Telegram/WhatsApp, re-ejecutar este notebook muestra cómo evoluciona la demanda de los usuarios.